# Using the processing configuration

## Choose the scan set

Set `SCAN_SET_ID` in the first code cell. This selects the sample,
scans, polarization pairs, and processing settings for this workup.

Each scan set has its own TOML file:

`config/processing/<SCAN_SET_ID>.toml`

For example:

`config/processing/ksr216_2026apr_longC_01.toml`

These paths are relative to the project folder. The CSV files identify
the scans and sample; the TOML file specifies how to process and review
them. Its settings apply across the polarization scans in that set.

## Main TOML sections

| Section | What it controls |
|---|---|
| `[darks]` | Dark-pixel percentile cutoff. Mask-sharing pairs are defined in `config/polarization_pairs.csv`. |
| `[beamstop]` | Beamstop rectangle center, length, width, and angle. |
| `[detector_borders]` | Number of pixels excluded along each detector edge. |
| `[center_mask]` | Saturation-search width and outer usable detector radius. |
| `[beam_centers."<pair_id>"]` | A configured center for each perpendicular pair, used for its center masks, fit starting points, and—after approval—integration. |
| `[beam_center_fit]` | Fit energy, search ranges, steps, target q, and fitting-specific masking settings. |
| `[runtime]` | Parallel jobs used for beam-center fitting. |
| `[integration]` | Postmask threshold for final integrated data. |
| `[normalization]` | Angular region used for half-ring normalization. |
| `[flux_correction]` | Whether Au mesh flux correction is enabled. |
| `[plotting]` | Energies displayed in individual-energy review plots. |

Edit values within the existing sections. TOML uses lowercase
`true` and `false`, and lists are written inside square brackets.
Do not add a second copy of an existing section header.

## Mask sharing

Mask-sharing pairs come from `config/polarization_pairs.csv`, using
the scan IDs and their configured polarizations—not hard-coded angles.

Dark-pixel masks and saturation masks are unioned only within each
perpendicular pair, such as 0°/90° or 45°/135°. Saturation masks and
center-mask radii are calculated separately at every energy.

The two scans in each pair share identical final masks. Different
pairs may have different masks.

The beamstop rectangle, detector borders, and outer usable radius
remain common to the scan set. Beam-center coordinates are shared
only within each configured perpendicular pair.

The current workflow requires each selected scan to belong to exactly
one configured perpendicular pair. It supports two-scan sets as well
as multiple pairs.

The old `union_across_polarizations` TOML setting is no longer used.

## Select energies to display

For example, to review only 284.8 eV:

`[plotting]`  
`energies_ev = [284.8]`

To review two energies:

`[plotting]`  
`energies_ev = [284.8, 286.0]`

The nearest measured energy is selected for each request. If multiple
requests select the same frame, that frame is shown only once.

- An explicit `energies_ev` list controls these plots for any scan type,
  including `repeats`.
- For `repeats`, omitting the list displays all energies.
- For `longC` and other non-`repeats` scan types, a nonempty list is required.

This selection changes display only. Calculations and exports still
include every energy.

Some figures intentionally behave differently: sector and aggregate
anisotropy heatmaps show the full energy axis; beam-center fit figures
show `fit_energy`; dark histograms use the dark frames; static-mask
reviews use images summed across energies.

## After editing the TOML

1. Save the TOML file with Ctrl+S.
2. Reload its settings in the notebook:

`configuration = load_scan_set(SCAN_SET_ID)`  
`processing = configuration["processing"]`

3. Rerun the affected cells.

Changing only `energies_ev` requires rerunning the relevant plotting
cells, not the calculations.

Changing processing settings requires rerunning the earliest affected
calculation and all dependent calculations afterward. Reloading the
configuration alone does not update existing results.

Changing a center under `[beam_centers."<pair_id>"]` requires rebuilding
the affected masks, refitting, reviewing consistency, and approving
that pair's center before reintegration.

When switching scan sets, restart the kernel and run the notebook in
order using the correct environment.

## Saving results

Export only after the results have been recalculated and reviewed using
the current settings. The loaded configuration values are recorded in
`manifest.json` alongside the Zarr store.

Each export creates a new timestamped folder, preserving earlier results.

Scan products, sector anisotropy, and pointwise anisotropy are saved
under `pairs/<pair_id>/`, separately for each polarization pair.
Each pair retains its own q grid and approved beam center.

Aggregate-anisotropy calculation, plotting, and export are currently
paused.

--------

In [ ]:
SCAN_SET_ID = "ksr216_2026apr_longC_01"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("dark_background")

In [ ]:
from pprint import pprint

from rsoxs_unified_workup.config import load_scan_set
from rsoxs_unified_workup.loading import (
    build_base_description,
    scan_ids_by_polarization,
)

configuration = load_scan_set(SCAN_SET_ID)

print(build_base_description(configuration["sample"]))
pprint(scan_ids_by_polarization(configuration))

In [ ]:
import httpx
import PyHyperScattering as phs
from tiled.client import from_uri

catalog = from_uri(
    "https://tiled.nsls2.bnl.gov",
    timeout=httpx.Timeout(None),
)

loader = phs.load.SST1RSoXSDB(
    corr_mode="none",
    catalog=catalog["rsoxs"]["raw"],
)

In [ ]:
from rsoxs_unified_workup.loading import load_scans

scan = load_scans(loader, configuration)

for polarization, scan_data in scan.items():
    print(
        f"Scan {scan_data['scan_id']}: "
        f"configured={polarization}°, "
        f"measured={scan_data['data'].attrs['polarization']}°"
    )

In [ ]:
from importlib import reload

from rsoxs_unified_workup.config import load_scan_set
import rsoxs_unified_workup.plotting as workup_plotting

configuration = load_scan_set(SCAN_SET_ID)
reload(workup_plotting)

if any(
    record["scan_set_id"] != SCAN_SET_ID
    for record in scan.values()
):
    raise ValueError(
        "Run the scan-loading cell for the selected scan set first."
    )

workup_plotting.plot_raw_frames(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
from rsoxs_unified_workup.darks import load_dark_frames

scan = load_dark_frames(loader, scan)

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_dark_histograms,
)

dark_upper_percentile = configuration[
    "processing"
]["darks"]["upper_percentile"]

dark_cutoffs = plot_dark_histograms(
    scan,
    upper_percentile=dark_upper_percentile,
)

{
    polarization: round(float(cutoff), 2)
    for polarization, cutoff in dark_cutoffs.items()
}

In [ ]:
from rsoxs_unified_workup.darks import build_dark_masks

dark_masks, dark_mask_summaries = build_dark_masks(
    scan,
    dark_cutoffs,
)

for polarization, summary in dark_mask_summaries.items():
    print(
        f"Scan {summary['scan_id']}, "
        f"polarization={polarization}°: "
        f"cutoff={summary['cutoff']:.2f}, "
        f"masked pixels={summary['masked_pixels']:,} "
        f"({summary['masked_percent']:.4f}%)"
    )

In [ ]:
from rsoxs_unified_workup.plotting import plot_dark_masks

plot_dark_masks(scan, dark_masks)

In [ ]:
from importlib import reload

import numpy as np
import rsoxs_unified_workup.config as workup_config
import rsoxs_unified_workup.masks as workup_masks

reload(workup_config)
reload(workup_masks)

configuration = workup_config.load_scan_set(SCAN_SET_ID)
processing = configuration["processing"]
mask_pairs = workup_config.get_mask_pairs(configuration)

detector_shape = next(iter(scan.values()))["data"].shape[-2:]

beamstop_mask, beamstop_vertices = workup_masks.create_beamstop_mask(
    detector_shape,
    processing["beamstop"],
)

border_mask = workup_masks.create_border_mask(
    detector_shape,
    processing["detector_borders"],
)

static_masks = workup_masks.combine_static_masks(
    dark_masks=dark_masks,
    beamstop_mask=beamstop_mask,
    border_mask=border_mask,
    mask_pairs=mask_pairs,
)

print("Mask-sharing pairs:")
for pair_id, (pol_1, pol_2) in mask_pairs.items():
    expected = (
        dark_masks[pol_1].transpose("pix_y", "pix_x").values
        | dark_masks[pol_2].transpose("pix_y", "pix_x").values
        | beamstop_mask
        | border_mask
    )

    for polarization in (pol_1, pol_2):
        np.testing.assert_array_equal(
            static_masks[polarization].values,
            expected,
        )

    print(f"  {pair_id}: {pol_1}° + {pol_2}°; pair union verified")

print("\nBeamstop vertices:")
for x, y in beamstop_vertices:
    print(f"  ({x:.2f}, {y:.2f})")

print("\nTotal masked pixels:")
for polarization, mask in static_masks.items():
    print(
        f"  polarization={polarization}°: "
        f"{int(mask.sum().item()):,}"
    )

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_static_mask_overlays,
)

plot_static_mask_overlays(
    scan,
    static_masks,
    beamstop_vertices,
)


In [ ]:
from importlib import reload

import numpy as np
import rsoxs_unified_workup.masks as workup_masks
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_masks)

scan = workup_masks.apply_static_masks(scan, static_masks)

saturation_masks = workup_masks.build_saturation_masks(
    scans=scan,
    mask_pairs=mask_pairs,
)

for pair_id, (pol_1, pol_2) in mask_pairs.items():
    mask = saturation_masks[pol_1]

    np.testing.assert_array_equal(
        mask.values,
        saturation_masks[pol_2].values,
    )

    print(
        f"\n{pair_id}: {pol_1}°/{pol_2}°; "
        f"dims={mask.dims}, shape={mask.shape}; "
        f"pair masks match"
    )

    energy_dimension = mask.energy.dims[0]
    review_indices = workup_plotting.review_energy_indices(
        scan_record=scan[pol_1],
        processing=configuration["processing"],
        energy_values=mask.energy.values,
    )

    for index in review_indices:
        frame = mask.isel({energy_dimension: index})
        energy = float(mask.energy.values[index])

        print(
            f"  Energy={energy:g} eV: "
            f"{int(frame.sum().item()):,} "
            f"saturated pixels across this pair"
        )


In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_saturation_overlays(
    scans=scan,
    saturation_masks=saturation_masks,
    processing=configuration["processing"],
)

In [ ]:
from importlib import reload

import numpy as np
import rsoxs_unified_workup.config as workup_config
import rsoxs_unified_workup.masks as workup_masks
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_config)
reload(workup_masks)

configuration = workup_config.load_scan_set(SCAN_SET_ID)
processing = configuration["processing"]
mask_pairs = workup_config.get_mask_pairs(configuration)
configured_beam_centers = workup_config.get_beam_centers(
    processing,
    mask_pairs,
)

center_masks, center_mask_summaries = (
    workup_masks.build_center_masks(
        saturation_masks=saturation_masks,
        beamstop_vertices=beamstop_vertices,
        processing=processing,
        mask_pairs=mask_pairs,
    )
)

for pair_id, (pol_1, pol_2) in mask_pairs.items():
    mask = center_masks[pol_1]

    np.testing.assert_array_equal(
        mask.values,
        center_masks[pol_2].values,
    )

    print(
        f"\n{pair_id}: {pol_1}°/{pol_2}°; "
        f"configured center={configured_beam_centers[pair_id]}; "
        f"dims={mask.dims}, shape={mask.shape}; "
        f"pair center masks match"
    )

    review_indices = workup_plotting.review_energy_indices(
        scan_record=scan[pol_1],
        processing=processing,
        energy_values=mask.energy.values,
    )

    for index in review_indices:
        summary = center_mask_summaries[pol_1][index]
        print(
            f"  Energy={summary['energy']:g} eV: "
            f"radius={summary['radius']:.2f} pixels, "
            f"source={summary['radius_source']}, "
            f"farthest point="
            f"({summary['farthest_x']:.2f}, "
            f"{summary['farthest_y']:.2f})"
        )

In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_center_mask_review(
    scans=scan,
    saturation_masks=saturation_masks,
    center_masks=center_masks,
    summaries=center_mask_summaries,
    processing=configuration["processing"],
)

In [ ]:
from importlib import reload

import numpy as np
import rsoxs_unified_workup.masks as workup_masks

reload(workup_masks)

final_masks = workup_masks.build_final_masks(
    static_masks=static_masks,
    saturation_masks=saturation_masks,
    center_masks=center_masks,
    mask_pairs=mask_pairs,
)

scan = workup_masks.apply_final_masks(scan, final_masks)

for polarization, final_mask in final_masks.items():
    energy_dimension = final_mask.energy.dims[0]
    mask_2d = final_mask.isel(
        {energy_dimension: 0}
    ).transpose("pix_y", "pix_x")

    print(
        f"Polarization={polarization}°: "
        f"full={final_mask.dims} {final_mask.shape}; "
        f"energy 0={mask_2d.dims} {mask_2d.shape}"
    )

for pair_id, (pol_1, pol_2) in mask_pairs.items():
    np.testing.assert_array_equal(
        final_masks[pol_1].values,
        final_masks[pol_2].values,
    )

    print(
        f"{pair_id}: {pol_1}°/{pol_2}°; "
        "final pair masks match"
    )


In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_final_masked_frames(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
from importlib import reload

import rsoxs_unified_workup.beam_center as beam_center

reload(beam_center)

median_center_suggestions = beam_center.fit_scan_set_beam_centers(
    scans=scan,
    processing=configuration["processing"],
    mask_pairs=mask_pairs,
)

for polarization, scan_record in scan.items():
    result = scan_record["beam_center_fit"]
    center = result["fine_best"]

    print(
        f"{polarization}°: "
        f"({center['x']}, {center['y']}), "
        f"cost={center['cost']:.6e}; "
        f"recorded passes={len(result['history'])}"
    )

print("\nMedian suggestions by pair:")
for pair_id, center in median_center_suggestions.items():
    print(f"  {pair_id}: {center}")


In [ ]:
import rsoxs_unified_workup.plotting as workup_plotting

for scan_record in scan.values():
    workup_plotting.plot_beam_center_fit(
        scan_record["beam_center_fit"],
        scan_record,
    )

In [ ]:
configuration = workup_config.load_scan_set(SCAN_SET_ID)
mask_pairs = workup_config.get_mask_pairs(configuration)

center_reviews = beam_center.check_beam_center_consistency(
    scans=scan,
    processing=configuration["processing"],
    mask_pairs=mask_pairs,
)

for pair_id, review in center_reviews.items():
    print(f"\n{pair_id}")
    print("Center used for mask:", review["center"])
    print("Fitted median:", review["median"])
    print("Difference, pixels:", review["difference"])
    print("Within fine-step tolerance:", review["within_tolerance"])

Each perpendicular polarization pair has its own configured center under
[beam_centers."<pair_id>"] in the processing TOML. That center is used
for the pair’s center masks, the starting point of each scan’s fit,
and—after explicit approval—the final integration.

Review each scan’s fit figures and each pair’s median consistency check.
If a pair’s median differs beyond the tolerance, update that pair’s
configured center, rebuild the center and final masks, and refit.

Approve each pair separately using the following cell.

In [ ]:
configuration = workup_config.load_scan_set(SCAN_SET_ID)
mask_pairs = workup_config.get_mask_pairs(configuration)

center_reviews = beam_center.check_beam_center_consistency(
    scans=scan,
    processing=configuration["processing"],
    mask_pairs=mask_pairs,
)

approved_beam_centers = {}

for pair_id, review in center_reviews.items():
    print(f"\n{pair_id}")
    print("Configured center:", review["center"])
    print("Fitted median:", review["median"])

    if not review["within_tolerance"]:
        raise ValueError(
            f"Update [beam_centers.{pair_id}] in the processing TOML, "
            "then rebuild the center and final masks and refit."
        )

    answer = input(
        f"After reviewing {pair_id}, type ACCEPT "
        "to approve this pair's center for integration: "
    )

    approved_beam_centers[pair_id] = beam_center.approve_beam_center(
        scans=scan,
        processing=configuration["processing"],
        mask_pairs=mask_pairs,
        pair_id=pair_id,
        approve=(answer.strip() == "ACCEPT"),
    )

    print("Approved pair center:", approved_beam_centers[pair_id])

In [ ]:
from importlib import reload
import rsoxs_unified_workup.integration as integration

reload(integration)

for polarization, scan_record in scan.items():
    scan_record["integration"] = integration.integrate_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )

    clean = scan_record["integration"]["clean"]
    print(
        f"{polarization}°: "
        f"dims={clean.dims}, shape={clean.shape}"
    )

In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

for scan_record in scan.values():
    workup_plotting.plot_integration_review(
        scan_record=scan_record,
        processing=configuration["processing"],
    )

In [ ]:
from importlib import reload
import rsoxs_unified_workup.integration as integration

reload(integration)

for polarization, scan_record in scan.items():
    scan_record["dark_integration"] = integration.integrate_dark_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )

    result = scan_record["dark_integration"]
    print(
        f"{polarization}°: "
        f"dark frame={result['dark_frame_index']}; "
        f"shape={result['darks_clean'].shape}"
    )

In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

for scan_record in scan.values():
    workup_plotting.plot_dark_integration_review(
        scan_record=scan_record,
        processing=configuration["processing"],
    )


In [ ]:
from importlib import reload

import rsoxs_unified_workup.normalization as normalization

reload(normalization)

for polarization, scan_record in scan.items():
    scan_record["normalization"] = normalization.normalize_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )
    result = scan_record["normalization"]

    full_check = result[
        "full_ring_normalized"
    ].mean("chi", skipna=True)

    half_region = normalization.select_chi_region(
        result["half_ring_normalized"],
        center_degrees=result[
            "half_ring_center_degrees"
        ],
        halfwidth_degrees=result[
            "half_ring_halfwidth_degrees"
        ],
    )
    half_check = half_region.mean(
        "chi",
        skipna=True,
    )

    full_deviation = np.abs(full_check - 1).max(
        skipna=True
    ).item()
    half_deviation = np.abs(half_check - 1).max(
        skipna=True
    ).item()

    print(
        f"{polarization}°: "
        f"full-ring maximum deviation={full_deviation:.3e}; "
        f"half-ring maximum deviation={half_deviation:.3e}"
    )


In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_normalization_review(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
import rsoxs_unified_workup.normalization as normalization

import rsoxs_unified_workup.plotting as workup_plotting

for polarization, scan_record in scan.items():
    scan_record["flux_correction"] = normalization.flux_correct_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )
    result = scan_record["flux_correction"]

    if not result:
        print(f"{polarization}°: flux correction disabled")
        continue

    flux = result["flux"]
    review_indices = workup_plotting.review_energy_indices(
        scan_record,
        configuration["processing"],
        flux.energy.values,
    )
    review_flux = flux.isel(energy=review_indices)

    print(
        f"{polarization}°: "
        f"corrected shape={result['flux_corrected'].shape}; "
        f"review energies={review_flux.energy.values}; "
        f"Au mesh current={review_flux.values}"
    )

    if review_flux.sizes["energy"] >= 2:
        energy_1, energy_2 = review_flux.energy.values[:2]
        ratio = review_flux.values[0] / review_flux.values[1]

        print(
            f"  Flux ratio ({energy_1:g}/{energy_2:g} eV)"
            f"={ratio:.6f}"
        )

In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_flux_correction_review(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
import rsoxs_unified_workup.anisotropy as anisotropy

anisotropy_results = anisotropy.calculate_anisotropy(
    configuration=configuration,
    scans=scan,
)

for sector in anisotropy_results["full_ring"].values():
    values = sector["anisotropy"]
    print(
        f"{sector['label']}: "
        f"parallel={sector['parallel_polarization']}°, "
        f"perpendicular="
        f"{sector['perpendicular_polarization']}°; "
        f"shape={values.shape}; "
        f"finite={np.isfinite(values).sum().item()}"
    )

In [ ]:
workup_plotting.plot_anisotropy_review(
    scans=scan,
    anisotropy_results=anisotropy_results,
)

In [ ]:
workup_plotting.plot_anisotropy_sector_overlays(
    scans=scan,
    configuration=configuration,
    anisotropy_results=anisotropy_results,
    energy_value=configuration["processing"][
        "beam_center_fit"
    ]["fit_energy"],
)

In [ ]:
# Aggregate methods are paused until their treatment is reviewed.
aggregate_anisotropy_results = None

# aggregate_anisotropy_results = (
#     anisotropy.aggregate_anisotropy(
#         anisotropy_results
#     )
# )
#
# for mode, results in aggregate_anisotropy_results.items():
#     print(f"\n{mode}")
#
#     for method_name in [
#         "method_1_global_intensity_ratio",
#         "method_2_pointwise_anisotropy",
#         "method_3_sector_anisotropy_mean",
#     ]:
#         values = results[method_name]
#
#         print(
#             f"  {method_name}: "
#             f"dims={values.dims}, "
#             f"shape={values.shape}, "
#             f"finite={np.isfinite(values.values).sum()}"
#         )

In [ ]:
# Aggregate comparison plotting is paused.
# workup_plotting.plot_aggregate_anisotropy_comparison(
#     scans=scan,
#     aggregate_results=aggregate_anisotropy_results,
# )

In [ ]:
from importlib import reload

import rsoxs_unified_workup.anisotropy as anisotropy

reload(anisotropy)

pointwise_anisotropy_results = (
    anisotropy.build_pointwise_anisotropy_maps(
        configuration=configuration,
        scans=scan,
    )
)

for mode, pair_results in (
    pointwise_anisotropy_results.items()
):
    print(f"\n{mode}")

    for pair_id, result in pair_results.items():
        values = result["anisotropy"]

        print(
            f"  {pair_id}: "
            f"dims={values.dims}, "
            f"shape={values.shape}, "
            f"chi range="
            f"{float(values.chi.min()):g} to "
            f"{float(values.chi.max()):g}; "
            f"finite="
            f"{np.isfinite(values.values).sum()}"
        )


In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_pointwise_anisotropy_review(
    scans=scan,
    pointwise_results=pointwise_anisotropy_results,
    processing=configuration["processing"],
)

In [ ]:
from datetime import datetime, timezone
from importlib import reload

import rsoxs_unified_workup.outputs as outputs

reload(outputs)

run_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

save_summary = outputs.save_workup_results(
    configuration=configuration,
    scans=scan,
    anisotropy_results=anisotropy_results,
    aggregate_anisotropy_results=None,
    pointwise_anisotropy_results=pointwise_anisotropy_results,
    output_root=f"results/runs/{run_id}",
    overwrite=False,
)

for key, value in save_summary.items():
    print(f"{key}: {value}")

In [ ]:
from importlib import reload

import rsoxs_unified_workup.outputs as outputs

reload(outputs)

outputs.print_workup_export_tree(save_summary)

In [ ]:
#notes to self (ignore these for now)

#make it so taht we can refit just a single pair rather than all four at once if we want to. this would be useful if we have a pair that is clearly off, but the other pair is fine. then we could just refit the one pair and not have to redo the other pair.
#eventually could make it so that masked individual pixels for union mask only apply for perp and para pairs of scans, i.e., only 0 annd 90 are a union, only 45 and 135 are a union
#could do the same with the center point, where 0 and 90 need to share a center, and 45 and 135 need to share a center, but all four don't need to be shared
#need to account for the fact that the value used for center mask is a guess at center, but is used to draw center radius. ideally would want to find real center before doing center mask. maybe some sort of way to update the center mask and reintegrate again after finding the fitted center
#regroup dictionaries
#probably want to thin kmore deeply about methods 1,2,3 and piontwise exports. what want to export? maybe just do pointwise, and everything beyond that is derived.
#q cap stuff: probably want to log qcap, but we don't stricly need to apply it so early on (before integration)... maybe think about this some more... don't want to throw away good high q data that pointwise anisotropy can catch
#include intensity visualizations (adn save them? or is remeshed enough? are we saving remesed right now? i thin kyes, it's "clean")
#also, think about how to do all of this stuff when we have a bunch of energys (long_C, long_O, etc)
#include integrated darks in final dictionaries
#do output smarter. zarr, just plain xarray output, etc
#account for some sort of raw images, maybe? thikning mostly of before dark subtraction. could be helpful for stuff when we haven't run enough darks
#half_ring_center_degrees and half_ring_halfwidth_degrees are usuelss i think. what could be mnore useful is knowing the min and max chi used for normalziation calculation
#decide smartly how to use flux when diong anisotropy calcs
#decide smartly how to do concatenation in a way that doesn't introduce chi mismatch
#make figures as an html file
#add commentary
#make some fields into user args (rather than buried in functions): 
    #peak q location for beam centering,
#check on empty (nan) vals in some matrix outputs in check statements
#probably pretty pointless to do whole ring normalization at all, right? actually not sure.
#make it so that integration progress is shown with progress bar rather than with  print statement at each pol/energy
#crazy, but is there a way to fit the energy-dependent beam center and fit each energy slice individually, to account for moving beam center?
#make so works with just two pols, or four pols, repeats or longC, flexibility in general
    #NPs vs not
    #longC, longO (maybe just make it "long")
#make all toml set up the same. also understandhow it works.        
#add markdown commentary/explanations
#when to reload in cells
#actually does it even make sense to have our multiple ways of calculating anisotropy? think it might make more sense to commit to pointwise and evaluate different "sectors" being different angular ranges. drop different A calc methods netirely
#anisotropy aggregation methods are being commented for now... will need to handle them later,particulary because indpenedent pairwise cnentesr mean inconsistent q grid.
#maybe do a darks integration without the postmask applied. either instead of or in addition to the postmask applied darks integration. this way we can see if the postmask is throwing away too much data, and maybe even use it to inform the postmask threshold.

#ticket add: switch to xray loading library from caitlyn/priyanka

#make some old legacy functions so that we can dmeonstrate how much better stuff has gotten via a timelapse